# Benchmark BamiBERT on PAP_NER Dataset (Multi-GPU T4x2)

This notebook is designed to be run on **Kaggle** (or Google Colab) to benchmark `Qualcomm-AI-Research/BamiBERT` on the `PhoBERT-Legal-NER` repository.
It automatically patches the repo to fully support Multi-GPU (DataParallel), fix gradient clipping bugs, and resolve PyTorch 2.6 weights compatibility.

### Cấu hình HuggingFace Token
Để tải BamiBERT nhanh hơn và không bị giới hạn rate limit, hãy cấu hình `HF_TOKEN` trong tab **Secrets** của Kaggle.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets!")
except Exception as e:
    print("Could not load Kaggle Secrets. Ensure HF_TOKEN is set if needed.")

In [ ]:
!rm -rf PhoBERT-Legal-NER
!git clone https://github.com/nghiemmmm/PhoBERT-Legal-NER
%cd PhoBERT-Legal-NER

### Bổ sung code hỗ trợ BamiBERT & Multi-GPU T4
Tạo model adapter cho BamiBERT, vá lỗi DataParallel, sửa gradient clipping và chuẩn hoá subword alignment.

In [ ]:
%%writefile patch_bamibert.py
import os
import re

# 1. Bổ sung BamiBertCrf Model (Trả về dict và padded tensor để DataParallel có thể gather)
bamibert_model_code = """\
from transformers import logging, RobertaForTokenClassification
from torchcrf import CRF
from .base import NerOutput
import torch
import torch.nn as nn

logging.set_verbosity_error()

class BamiBertCrf(RobertaForTokenClassification):
    def __init__(self, config):
        super(BamiBertCrf, self).__init__(config=config)
        self.num_labels = config.num_labels
        self.crf = CRF(config.num_labels, batch_first=True)
        self.init_weights()

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None, valid_ids=None, label_masks=None):
        base_model = getattr(self, self.base_model_prefix)
        seq_outputs = base_model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            head_mask=None
        )[0]
        
        batch_size, max_len, feat_dim = seq_outputs.shape
        range_vector = torch.arange(0, batch_size, dtype=torch.long, device=seq_outputs.device).unsqueeze(1)
        seq_outputs = seq_outputs[range_vector, valid_ids]
        
        seq_outputs = getattr(self, "dropout", nn.Dropout(0.1))(seq_outputs)
        logits = self.classifier(seq_outputs)
        
        seq_tags = self.crf.decode(logits, mask=label_masks != 0)
        
        padded_tags = torch.zeros_like(label_masks)
        for i, tags in enumerate(seq_tags):
            padded_tags[i, :len(tags)] = torch.tensor(tags, dtype=torch.long, device=logits.device)
            
        if labels is not None:
            log_likelihood = self.crf(logits, labels, mask=label_masks.bool())
            return {"loss": -1.0 * log_likelihood, "tags": padded_tags}
        else:
            return {"tags": padded_tags}
"""
with open("vphoberttagger/models/bamibert.py", "w", encoding="utf-8") as f:
    f.write(bamibert_model_code)

with open("vphoberttagger/models/__init__.py", "a", encoding="utf-8") as f:
    f.write("\nfrom .bamibert import BamiBertCrf\n")

# Vá PhoBertCrf gốc để cũng trả về dict cho DataParallel và đổi mask sang bool()
with open("vphoberttagger/models/phobert.py", "r", encoding="utf-8") as f:
    phobert_content = f.read()

old_block = """\
        if labels is not None:
            log_likelihood = self.crf(logits, labels, mask=label_masks.type(torch.uint8))
            return NerOutput(loss=-1.0 * log_likelihood, tags=seq_tags)
        else:
            return NerOutput(tags=seq_tags)"""

new_block = """\
        padded_tags = torch.zeros_like(label_masks)
        for i, tags in enumerate(seq_tags):
            padded_tags[i, :len(tags)] = torch.tensor(tags, dtype=torch.long, device=logits.device)
            
        if labels is not None:
            log_likelihood = self.crf(logits, labels, mask=label_masks.bool())
            return {"loss": -1.0 * log_likelihood, "tags": padded_tags}
        else:
            return {"tags": padded_tags}"""

phobert_content = phobert_content.replace(old_block, new_block)
with open("vphoberttagger/models/phobert.py", "w", encoding="utf-8") as f:
    f.write(phobert_content)

# 2. Cập nhật constant.py để mapping BamiBERT
with open("vphoberttagger/constant.py", "r", encoding="utf-8") as f:
    constant_content = f.read()

constant_content = constant_content.replace("from vphoberttagger.models import PhoBertCrf", "from vphoberttagger.models import PhoBertCrf, BamiBertCrf")
constant_content = constant_content.replace("'vinai/phobert-large': convert_word_segment_examples_features,", "'vinai/phobert-large': convert_word_segment_examples_features,\n    'Qualcomm-AI-Research/BamiBERT': convert_word_segment_examples_features,")
constant_content = constant_content.replace("'vinai/phobert-large': {\n        'crf': PhoBertCrf,\n    }", "'vinai/phobert-large': {\n        'crf': PhoBertCrf,\n    },\n    'Qualcomm-AI-Research/BamiBERT': {\n        'crf': BamiBertCrf,\n    }")
with open("vphoberttagger/constant.py", "w", encoding="utf-8") as f:
    f.write(constant_content)

# 3. Vá lỗi trong processor.py (xoá ffill vì phá vỡ cấu trúc câu trống, sửa subword alignment chuẩn)
with open("vphoberttagger/processor.py", "r", encoding="utf-8") as f:
    proc_content = f.read()

proc_content = proc_content.replace('data.fillna(method="ffill")', '')

new_processor_func = """\
def convert_word_segment_examples_features(data_path: Union[str, os.PathLike], tokenizer, label2id, header_names: List[str], max_seq_len: int = 256, use_crf: bool = True) -> List[NerFeatures]:
    features = []
    tokens = []
    tag_ids = []
    
    data = pd.read_csv(data_path, sep=r'\\s+', encoding='utf-8', skip_blank_lines=False, names=header_names)
    
    for row_idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Load dataset {data_path}..."):
        if pd.notna(row.token):
            tokens.append(str(row.token).strip().replace(' ', '_'))
            tag_ids.append(label2id.index(str(row.ner).strip()))
            if not row_idx == len(data) - 1:
                continue
                
        seq_len = len(tokens)
        if seq_len == 0:
            continue
            
        encoding = tokenizer(tokens, is_split_into_words=True, padding='max_length', truncation=True, max_length=max_seq_len)
        word_ids = encoding.word_ids()
        
        valid_ids = np.zeros(len(encoding.input_ids), dtype=int)
        label_marks = np.zeros(len(encoding.input_ids), dtype=int)
        valid_labels = np.ones(len(encoding.input_ids), dtype=int) * -100
        
        previous_word_idx = None
        i = 0
        for idx, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != previous_word_idx:
                if use_crf:
                    valid_ids[i] = idx
                else:
                    valid_ids[idx] = 1
                valid_labels[idx] = tag_ids[word_idx]
                i += 1
            previous_word_idx = word_idx
            
        actual_seq_len = i
        label_marks[:actual_seq_len] = 1
        final_tag_ids = [tag_ids[w_idx] for w_idx in range(actual_seq_len)]
        final_tag_ids.extend([0] * (max_seq_len - actual_seq_len))
            
        items = {key: val for key, val in encoding.items()}
        if 'token_type_ids' not in items:
            items['token_type_ids'] = [0] * max_seq_len
        items['labels'] = final_tag_ids if use_crf else valid_labels
        items['valid_ids'] = valid_ids
        items['label_masks'] = label_marks if use_crf else valid_ids
        features.append(NerFeatures(**items))

        tokens = []
        tag_ids = []
        
    return features
"""

start_idx = proc_content.find("def convert_word_segment_examples_features")
if start_idx != -1:
    proc_content = proc_content[:start_idx] + new_processor_func
with open("vphoberttagger/processor.py", "w", encoding="utf-8") as f:
    f.write(proc_content)

# 4. Cấu hình lại toàn bộ trainer.py để chạy được DataParallel (Multi-GPU), sửa Gradient Bug
with open("vphoberttagger/trainer.py", "r", encoding="utf-8") as f:
    trainer_content = f.read()

# Fix hardcoded model_clss so BamiBertCrf is ACTUALLY used instead of PhoBertCrf!
trainer_content = trainer_content.replace(
    "model_clss = PhoBertCrf",
    "from vphoberttagger.constant import MODEL_MAPPING\n    model_clss = MODEL_MAPPING[args.model_name_or_path]['crf'] if 'crf' in args.model_arch else MODEL_MAPPING[args.model_name_or_path]['softmax']"
)

# 4a. Fix Gradient Clipping Order (NaN bug) and dictionary loss access
buggy_clip = """\
        outputs = model(**batch)
        # backward pass
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
        optim.zero_grad()
        outputs.loss.backward()
        optim.step()"""
fixed_clip = """\
        outputs = model(**batch)
        loss = outputs['loss'].sum() if isinstance(outputs, dict) else outputs.loss
        # backward pass
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
        optim.step()"""
trainer_content = trainer_content.replace(buggy_clip, fixed_clip)
trainer_content = trainer_content.replace("tr_loss += outputs.loss.detach().item()", "tr_loss += loss.detach().item()")

# 4b. Fix Evaluate process for Dict/Padded Tags
eval_old = """\
            outputs = model(**batch)
            eval_loss += outputs.loss.detach().item()
            active_accuracy = batch['label_masks'].view(-1) != 0
            labels = torch.masked_select(batch['labels'].view(-1), active_accuracy)
            eval_golds.extend(labels.detach().cpu().tolist())
            if isinstance(outputs.tags[-1], list):
                eval_preds.extend(list(itertools.chain(*outputs.tags)))
            else:
                eval_preds.extend(outputs.tags)
"""
eval_new = """\
            outputs = model(**batch)
            loss = outputs['loss'].sum() if isinstance(outputs, dict) else outputs.loss
            eval_loss += loss.detach().item()
            active_accuracy = batch['label_masks'].view(-1) != 0
            labels = torch.masked_select(batch['labels'].view(-1), active_accuracy)
            eval_golds.extend(labels.detach().cpu().tolist())
            
            preds = outputs['tags'] if isinstance(outputs, dict) else outputs.tags
            if isinstance(preds, torch.Tensor):
                pred_labels = torch.masked_select(preds.view(-1), active_accuracy)
                eval_preds.extend(pred_labels.detach().cpu().tolist())
            elif isinstance(preds[-1], list):
                eval_preds.extend(list(itertools.chain(*preds)))
            else:
                eval_preds.extend(preds)
"""
trainer_content = trainer_content.replace(eval_old, eval_new)

# 4c. Setup DataParallel in Train
dp_setup = """\
    if args.load_weights is not None:
        LOGGER.info(f'Load pretrained model weights from "{args.load_weights}"')
        if device == 'cpu':
            checkpoint_data = torch.load(args.load_weights, map_location='cpu', weights_only=False)
        else:
            checkpoint_data = torch.load(args.load_weights, weights_only=False)
        model.load_state_dict(checkpoint_data['model'])
        checkpoint_data = None

    if torch.cuda.device_count() > 1:
        LOGGER.info(f"Using {torch.cuda.device_count()} GPUs for training!")
        model = torch.nn.DataParallel(model)
"""
old_load_weights = """\
    if args.load_weights is not None:
        LOGGER.info(f'Load pretrained model weights from "{args.load_weights}"')
        if device == 'cpu':
            checkpoint_data = torch.load(args.load_weights, map_location='cpu')
        else:
            checkpoint_data = torch.load(args.load_weights)
        model.load_state_dict(checkpoint_data['model'])
        checkpoint_data = None
"""
trainer_content = trainer_content.replace(old_load_weights, dp_setup)

# 4d. Fix hardcoded optimizer layers for unwrapped model
old_logic = r"if\s+args\.model_name_or_path\s*==\s*'vinai/phobert-base':\s*\n\s*bert_param_optimizer\s*=\s*list\(model\.roberta\.named_parameters\(\)\)\s*\n\s*else:\s*\n\s*bert_param_optimizer\s*=\s*list\(model\.bert\.named_parameters\(\)\)\s*\n\s*ner_param_optimizer\s*=\s*list\(model\.classifier\.named_parameters\(\)\)\s*\n\s*if\s+'lstm'\s+in\s+args\.model_arch:\s*\n\s*ner_param_optimizer\.extend\(list\(model\.lstm\.named_parameters\(\)\)\)\s*\n\s*if\s+'crf'\s+in\s+args\.model_arch:\s*\n\s*ner_param_optimizer\.extend\(list\(model\.crf\.named_parameters\(\)\)\)"
new_logic = """unwrap_model = model.module if hasattr(model, 'module') else model
    if hasattr(unwrap_model, 'base_model_prefix'):
        bert_param_optimizer = list(getattr(unwrap_model, unwrap_model.base_model_prefix).named_parameters())
    elif hasattr(unwrap_model, 'roberta'):
        bert_param_optimizer = list(unwrap_model.roberta.named_parameters())
    else:
        bert_param_optimizer = list(unwrap_model.bert.named_parameters())
    ner_param_optimizer = list(unwrap_model.classifier.named_parameters())
    if 'lstm' in args.model_arch:
        ner_param_optimizer.extend(list(unwrap_model.lstm.named_parameters()))
    if 'crf' in args.model_arch:
        ner_param_optimizer.extend(list(unwrap_model.crf.named_parameters()))"""
trainer_content = re.sub(old_logic, new_logic, trainer_content)

# 4e. Fix save model state dict to use unwrapped model
trainer_content = trainer_content.replace(
    "'model': model.state_dict()",
    "'model': (model.module if hasattr(model, 'module') else model).state_dict()"
)

with open("vphoberttagger/trainer.py", "w", encoding="utf-8") as f:
    f.write(trainer_content)

# 5. Vá lỗi weights_only=True trong PyTorch 2.6 cho torch.load
def patch_torch_load(filepath):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        content = content.replace("torch.load(cached_path)", "torch.load(cached_path, weights_only=False)")
        content = content.replace("torch.load(args.load_weights)", "torch.load(args.load_weights, weights_only=False)")
        content = content.replace("torch.load(model_path, map_location=device)", "torch.load(model_path, map_location=device, weights_only=False)")
        content = content.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)

patch_torch_load("vphoberttagger/dataset.py")
patch_torch_load("vphoberttagger/trainer.py")
patch_torch_load("vphoberttagger/predictor.py")

print("All patches applied successfully!")


In [ ]:
!python patch_bamibert.py

### Xoá Cache dữ liệu bị lỗi cũ (Chỉ chạy 1 lần)

In [ ]:
!rm -rf Dataset/*.cached

### Cài đặt thư viện

In [ ]:
!pip install -r requirements.txt
# Đôi khi cài thêm pytorch-crf để chắc chắn
!pip install pytorch-crf

### Huấn luyện và Đánh giá (Benchmark)
Quá trình này sẽ sử dụng HF_TOKEN đã cài đặt để tải weights nhanh nhất.

In [ ]:
!python main.py train \
    --task pap_ner \
    --data_dir ./Dataset \
    --model_name_or_path Qualcomm-AI-Research/BamiBERT \
    --model_arch crf \
    --output_dir outputs \
    --max_seq_length 256 \
    --train_batch_size 32 \
    --eval_batch_size 32 \
    --learning_rate 2e-5 \
    --classifier_learning_rate 1e-3 \
    --epochs 20 \
    --early_stop 3 \
    --run_test